
# NeoOLAF native DocRED layer ablation — Skai TV v3

This patch tests one DocRED document through the complete native NeoOLAF Layer 0–12 pipeline.

Changes from v2:

- explicit input-level relation specification, direction rules, and synthetic examples;
- no gold entities or gold relation pairs in the pipeline input;
- relation candidates must link to canonical `Pxxx : label` ontology properties;
- relation schemas are promoted while named instances remain instances;
- Layer 2 runs with 12 workers and a small output budget;
- the existing Layer 4 relation task runs concurrently with 8 workers;
- per-layer timeouts and response caps prevent one malformed response from blocking the run;
- parse failures, response-cap violations, API errors, Layer 4 errors, prompts, responses, retrievals, and layer states are saved;
- strict cumulative evaluation is produced after every layer.

No file under `src/neoolaf` is modified. No direct DocRED extractor, source-entity anchoring, closure rule, or gold-derived fact is used.


In [ ]:

from __future__ import annotations

import json
import os
import sys
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from docred_native_ablation_v3 import (
    analyze_run,
    load_layer_states,
    read_json,
    read_jsonl,
    run_native_pipeline,
)

print("PROJECT_ROOT =", PROJECT_ROOT)


## 1. Configuration

In [ ]:

INPUT_JSONL = NOTEBOOK_DIR / "data/docred_skai_tv_input.jsonl"
GOLD_JSONL = NOTEBOOK_DIR / "data/docred_skai_tv_gold.jsonl"
ONTOLOGY_PATH = NOTEBOOK_DIR / "ontology/docred_redocred_neoolaf_compatible.ttl"
ONTOLOGY_ORIGINAL = NOTEBOOK_DIR / "ontology/docred_redocred_original.ttl"
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/docred_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/docred_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/docred_profile_native_ablation_v3.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_docred_native_ablation_v3.json"

RUNS_ROOT = NOTEBOOK_DIR / "runs/docred_native_layer_ablation"
RUN_DIR = RUNS_ROOT / "skai_tv_guided_parallel_v3"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")

# Layer 2 uses 12 workers; Layer 4 uses 8 according to the profile.
WORKERS = 12
REASONING_EFFORT = "minimal"
RUN_PIPELINE = True
CLEAN_RUN_DIR = True

print("Input:", INPUT_JSONL)
print("Profile:", PROFILE_PATH)
print("Guidance:", GUIDANCE_PATH)
print("Run dir:", RUN_DIR)
print("Model:", MODEL_NAME)
print("API key available:", bool(API_KEY))


## 2. Preflight and explicit input guidance

In [ ]:

required = [
    INPUT_JSONL, GOLD_JSONL, ONTOLOGY_PATH, ONTOLOGY_ORIGINAL,
    RELATION_CATALOG, RELATION_ALIASES, PROFILE_PATH, GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

input_rows = read_jsonl(INPUT_JSONL)
gold_rows = read_jsonl(GOLD_JSONL)
assert len(input_rows) == 1 and len(gold_rows) == 1
assert "entities" not in input_rows[0] and "relations" not in input_rows[0]
assert input_rows[0]["document_id"] == gold_rows[0]["document_id"]

profile = read_json(PROFILE_PATH)
guidance = read_json(GUIDANCE_PATH)
task_guidance = input_rows[0].get("task_guidance", {})

print("Document:", input_rows[0]["document_id"], "-", input_rows[0]["title"])
print("Source characters:", len(input_rows[0]["text"]))
print("Whole-document input:", profile["chunking"]["whole_document_preferred"])
print("Profile:", profile["profile_name"])
print("Explicit target relation IDs:", len(task_guidance.get("allowed_relation_ids", [])))
print("Synthetic relation examples:", len(task_guidance.get("relation_examples", [])))
print("Layer 2 workers:", profile["layers"]["layer02_candidate_enrichment"]["max_concurrency"])
print("Layer 4 workers:", profile["layers"]["layer04_candidate_relation_extraction"]["max_concurrency"])
print("Layer 2 output tokens:", profile["layers"]["layer02_candidate_enrichment"]["max_output_tokens"])
print("Layer 4 output tokens:", profile["layers"]["layer04_candidate_relation_extraction"]["max_output_tokens"])

display(pd.DataFrame(task_guidance.get("relation_specs", [])))
display(pd.DataFrame(task_guidance.get("relation_examples", [])))

display(Markdown(
    "**Anti-leakage check:** task guidance is metadata merged into `UserGuidance`; it is not "
    "prepended to the source article and contains no gold entity clusters or gold relation pairs."
))


## 3. Run full native pipeline

In [ ]:

if RUN_PIPELINE:
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    final_state = run_native_pipeline(
        project_root=PROJECT_ROOT,
        input_jsonl=INPUT_JSONL,
        ontology_path=ONTOLOGY_PATH,
        profile_path=PROFILE_PATH,
        guidance_path=GUIDANCE_PATH,
        relation_catalog_path=RELATION_CATALOG,
        relation_aliases_path=RELATION_ALIASES,
        run_dir=RUN_DIR,
        model_name=MODEL_NAME,
        api_key=API_KEY,
        host=OPENROUTER_HOST,
        workers=WORKERS,
        reasoning_effort=REASONING_EFFORT,
        verbose=True,
        clean_run_dir=CLEAN_RUN_DIR,
    )
    print("Full native run completed.")
else:
    print("RUN_PIPELINE=False: reusing", RUN_DIR)



### Runtime artifacts

The run saves:

- `run_manifest.json`, `effective_user_guidance.json`, and `input_task_guidance.json`;
- `run_logs/console.log`;
- `run_logs/llm_calls.jsonl`, `llm_errors.jsonl`, `llm_parse_errors.jsonl`, and `llm_response_cap_errors.jsonl`;
- `run_logs/layer04_relation_errors.jsonl`;
- `run_logs/ontology_retrieval.jsonl`;
- raw responses under `run_logs/responses/`;
- complete state, output, metadata, prompt, and timing artifacts for every layer;
- restart checkpoint and Layer 12 exports.


## 4. Layer-by-layer ablation and strict evaluation

In [ ]:

summary = analyze_run(
    run_dir=RUN_DIR,
    gold_jsonl=GOLD_JSONL,
    catalog_path=RELATION_CATALOG,
    aliases_path=RELATION_ALIASES,
)

layer_df = pd.DataFrame(summary["layer_summary"])
display(layer_df)

print("Strict final native evaluation")
display(pd.DataFrame([{
    key: summary["strict_evaluation"][key]
    for key in [
        "predicted", "gold", "true_positive", "false_positive",
        "false_negative", "precision", "recall", "f1",
    ]
}]))

print("Cumulative strict evaluation")
cumulative_df = pd.DataFrame(summary["cumulative_strict_evaluation"])
display(cumulative_df)

print("First-failure counts")
print(summary["failure_counts_v3"])


## 5. Relation-instance trace

In [ ]:

trace_df = pd.read_csv(RUN_DIR / "analysis/gold_relation_trace_v3.csv")
display(trace_df)

display(
    trace_df.groupby("first_failure", dropna=False)
    .size()
    .reset_index(name="gold_relations")
    .sort_values("gold_relations", ascending=False)
)


## 6. Native relation canonicalization and ontology promotion

In [ ]:

states = {index: state for index, _, state in load_layer_states(RUN_DIR)}
layer3 = states.get(3)
layer4 = states.get(4)
layer5 = states.get(5)
layer6 = states.get(6)
layer8 = states.get(8)
layer11 = states.get(11)

if layer3:
    relation_rows = []
    for candidate in layer3.relation_candidates or []:
        relation_rows.append({
            "candidate_id": candidate.candidate_id,
            "canonical_label": candidate.canonical_label,
            "mentions": [m.text for m in candidate.mentions],
            "aliases": candidate.aliases,
            "controlled_relation_hints": [
                h for h in candidate.ontology_hints
                if str(h).lower().startswith("controlled_relation:")
            ],
            "promote_hints": [
                h for h in candidate.ontology_hints
                if str(h).lower().startswith("promote_to_ontology:")
            ],
        })
    display(pd.DataFrame(relation_rows))

if layer4:
    print("Layer 4 assertions")
    display(pd.DataFrame([{
        "assertion_id": x.assertion_id,
        "source": x.source_candidate_label,
        "predicate": x.relation_label,
        "target": x.target_candidate_label,
        "confidence": x.confidence,
        "justification": x.justification,
    } for x in layer4.candidate_relation_assertions or []]))

if layer5:
    print("Layer 5 native triples")
    display(pd.DataFrame([{
        "triple_id": x.triple_id,
        "subject": x.subject_label,
        "predicate": x.predicate_label,
        "object": x.object_label,
        "confidence": x.confidence,
    } for x in layer5.candidate_triples or []]))

print("Ontology-stage counts")
display(pd.DataFrame([{
    "ontology_relation_candidates_L6": len(layer6.ontology_relation_candidates or []) if layer6 else 0,
    "axiom_schema_candidates_L8": len(layer8.axiom_schema_candidates or []) if layer8 else 0,
    "completion_candidates_L11": len(layer11.completion_candidates or []) if layer11 else 0,
}]))


## 7. Separate native, canonical, strict, and not-in-gold views

In [ ]:

for filename in [
    "native_lexical_triples.csv",
    "ontology_canonical_triples.csv",
    "strict_docred_predictions.csv",
    "predictions_not_in_gold_manual_review.csv",
]:
    path = RUN_DIR / "analysis" / filename
    print("\n", filename)
    if path.is_file():
        display(pd.read_csv(path))
    else:
        print("No rows/file generated.")


## 8. Speed, parallelism, retrieval, and errors

In [ ]:

def load_optional_jsonl(path: Path) -> pd.DataFrame:
    if not path.is_file():
        return pd.DataFrame()
    return pd.DataFrame(read_jsonl(path))

calls = load_optional_jsonl(RUN_DIR / "run_logs/llm_calls.jsonl")
errors = load_optional_jsonl(RUN_DIR / "run_logs/llm_errors.jsonl")
parse_errors = load_optional_jsonl(RUN_DIR / "run_logs/llm_parse_errors.jsonl")
cap_errors = load_optional_jsonl(RUN_DIR / "run_logs/llm_response_cap_errors.jsonl")
layer4_errors = load_optional_jsonl(RUN_DIR / "run_logs/layer04_relation_errors.jsonl")
retrieval = load_optional_jsonl(RUN_DIR / "run_logs/ontology_retrieval.jsonl")

if not calls.empty:
    display(calls)
    display(calls.groupby("layer_tag").agg(
        calls=("call_index", "count"),
        total_recorded_seconds=("elapsed_seconds", "sum"),
        maximum_call_seconds=("elapsed_seconds", "max"),
        mean_response_chars=("response_chars", "mean"),
    ).reset_index())

print("API/backend errors:", len(errors))
if not errors.empty: display(errors)
print("JSON parse errors:", len(parse_errors))
if not parse_errors.empty: display(parse_errors)
print("Response-cap violations:", len(cap_errors))
if not cap_errors.empty: display(cap_errors)
print("Layer 4 relation failures:", len(layer4_errors))
if not layer4_errors.empty: display(layer4_errors)

if not retrieval.empty:
    display(retrieval)
    display(retrieval.groupby("layer_name").size().reset_index(name="retrieval_calls"))



## Interpretation checklist

After execution, verify:

1. Layer 1 extracted fewer peripheral phrases while retaining all relevant endpoints and relation spans.
2. Layer 2 relation expressions contain `controlled_relation:Pxxx : label`.
3. Layer 3 relation mentions have canonical P-identifiers, while profile schema candidates remain mention-free.
4. Layer 4 runtime is concurrent rather than sequential.
5. Layer 6 contains non-zero ontology relation candidates.
6. The cumulative strict table shows exactly where precision/recall first changes.
7. Predictions absent from gold are treated as manual-review candidates, not automatically claimed as correct.
